# Subject Fatigue
We examine whether fatigue has an effect on LWS probability (i.e., $P[\text{miss} | \text{target}]$).<br>
To examine this we use the trial number as a proxy for fatigue, including only valid trials in the analysis, but counting all trials when calculating the trial number (i.e., invalid trials contribute to fatigue but not to the LWS probability).

In [1]:
import os

import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

import config as cnfg
from analysis.helpers.funnels.build_funnels import build_event_classification_funnel

pio.renderers.default = "notebook"      # "notebook" or "browser"

In [2]:
funnel_results = build_event_classification_funnel(
    cnfg.OUTPUT_PATH, "lws", "visit", exclude="invalid_trials"
)

### Visualize

In [6]:
subject_stats = (
    funnel_results
    .groupby(['subject', 'trial'], observed=True)
    .agg(
        lws_ratio=('is_lws', 'mean'),
        total_visits=('is_lws', 'count')
    )
    .reset_index(drop=False)
)
overall_stats = (
    subject_stats
    .groupby('trial')
    .agg(
        lws_ratio=('lws_ratio', 'mean'),
        total_visits=('total_visits', 'mean'),
        visits_sem=('total_visits', 'sem'),
    )
    .reset_index(drop=False)
)

# Dual-Axis Figure
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Total Visits (Bars or faint line)
fig.add_trace(
    go.Bar(
        x=overall_stats['trial'],
        y=overall_stats['total_visits'],
        error_y=dict(
            type='data',
            array=overall_stats['visits_sem'],
            visible=True,
            color='gray',
            thickness=2,
            width=3,
        ),
        name="Total Target Visits",
        marker=dict(color='lightgray', line=dict(color='gray', width=1)),
        opacity=0.5
    ), secondary_y=True
)

# # individual LWS Probability (spagetti lines)
# for sub in subject_stats['subject'].unique():
#     sub_data = subject_stats[subject_stats['subject'] == sub]
#     fig.add_trace(
#         go.Scatter(
#             x=sub_data['trial'],
#             y=sub_data['lws_ratio'],
#             name=f"Subject {sub} - LWS Probability",
#             mode='lines', line=dict(color='red', width=1),
#             opacity=0.15,
#             showlegend=False, hoverinfo='skip'
#         ),
#         secondary_y=False
#     )

# Overall LWS Probability (Bold line)
fig.add_trace(
    go.Scatter(
        x=overall_stats['trial'],
        y=overall_stats['lws_ratio'],
        name="LWS Probability",
        mode='lines+markers', line=dict(color='red', width=3)
    ), secondary_y=False
)

fig.update_xaxes(
    title=dict(text="Trial Number", font=cnfg.AXIS_LABEL_FONT),
    tickvals=[1, 10, 20, 30, 40, 50, 60],
    tickfont=cnfg.AXIS_TICK_FONT,
    showgrid=False, zeroline=False,
)
fig.update_yaxes(
    secondary_y=False,
    title=dict(text="Probability of LWS Error", font=cnfg.AXIS_LABEL_FONT),
    range=[-0.01, 1.01],
    tickvals=np.arange(0, 1.01, 0.25), tickfont=cnfg.AXIS_TICK_FONT,
    showgrid=False, zeroline=False,
)
fig.update_yaxes(
    secondary_y=True,
    title=dict(text="Total Visits", font=cnfg.AXIS_LABEL_FONT),
    tickfont=cnfg.AXIS_TICK_FONT,
    showgrid=False, zeroline=False,
)
fig.update_layout(
    width=900, height=500,
    title=dict(text="LWS Dynamics for Time on Task", font=cnfg.TITLE_FONT),
    legend=dict(
        orientation="h",
        yanchor="middle", y=1,
        xanchor="right", x=0.95,
    ),
    margin=dict(t=40, b=40, l=40, r=10),
    template="plotly_white",
)

fig.show()

### Statistical Analysis
To statistically analyze the effect of trial number on LWS probability, we use the R package `mgcv` to fit a Generalized Additive Model (GAM) with a smooth term for trial number, while controlling for trial type and accounting for subject-level random effects. The model is specified as follows:
$$ \text{logit}(P[\text{miss} | \text{on target}]) \sim C(\text{trial\_category}) + s(\text{trial number}) + (1|\text{subject}) $$
The smooth term $s(\text{trial number})$ allows us to capture potential non-linear effects of trial number on LWS probability, while the categorical term $C(\text{trial\_category})$ controls for differences in trial types (e.g., target present vs. target absent), and the random effect $(1|\text{subject})$ accounts for variability across subjects.
<br><br>
#### Prediction and Visualization
The analysis itself is conducted in R due to the poor Python --> R engine avialble with `rpy2`. Here, we visualize the predicted values from the fitted GAM model.

In [4]:
def plot_temporal_predictions(csv_path: str) -> go.Figure:
    # read and aggregate the data
    df = pd.read_csv(csv_path, index_col=None)
    subj_agg = df.groupby(['subject', 'trial'])['prob'].mean().reset_index()  # first aggregate within subject
    pop_stats = (  # then aggregate across subjects
        subj_agg
        .groupby(["trial"])
        .agg(mean_prob=('prob', 'mean'), sem_prob=('prob', 'sem'))
        .reset_index()
    )

    # calculate 95% CIs
    pop_stats['upper_ci'] = pop_stats['mean_prob'] + (1.96 * pop_stats['sem_prob'])
    pop_stats['lower_ci'] = pop_stats['mean_prob'] - (1.96 * pop_stats['sem_prob'])

    # generate the figure
    preds_fig = go.Figure()
    preds_fig.add_trace(go.Scatter(  # 95% CI ribbon
        x=pd.concat([pop_stats['trial'], pop_stats['trial'].iloc[::-1]]),
        y=pd.concat([pop_stats['upper_ci'], pop_stats['lower_ci'].iloc[::-1]]),
        name="95% CI (Across Subjects)", showlegend=False,
        fill='toself',
        fillcolor='rgba(0,0,0, 0.2)',
        line=dict(color='rgba(0,0,0,0.25)'),
        hoverinfo="skip",
    ))
    preds_fig.add_trace(go.Scatter(  # population mean
        x=pop_stats['trial'],
        y=pop_stats['mean_prob'],
        name="Predicted Trend", showlegend=True,
        mode='lines',
        line=dict(color='black', width=3),
    ))

    # # Optional: Add faint individual subject lines (Spaghetti plot)
    # for sub in subj_agg['subject'].unique():
    #     sub_df = subj_agg[subj_agg['subject'] == sub]
    #     preds_fig.add_trace(go.Scatter(
    #         x=sub_df['trial'],
    #         y=sub_df['prob'],
    #         name=f"Subject {sub}", showlegend=False,
    #         mode='lines',
    #         line=dict(color='black', width=1),
    #         opacity=0.1,
    #         # hoverinfo='skip'
    #     ))

    # Formatting
    preds_fig.update_xaxes(
        title=dict(text="Trial Number", font=cnfg.AXIS_LABEL_FONT),
        tickvals=[1, 10, 20, 30, 40, 50, 60], tickfont=cnfg.AXIS_TICK_FONT,
        showgrid=False, zeroline=False,
    )
    preds_fig.update_yaxes(
        title=dict(text="Predicted P[LWS]", font=cnfg.AXIS_LABEL_FONT),
        # range=[min(pop_stats['lower_ci']) * 0.9, max(pop_stats['upper_ci']) * 1.05],
        range=[-0.01, 1.01],
        tickvals=np.arange(0, 1.01, 0.25), tickfont=cnfg.AXIS_TICK_FONT,
        showgrid=True, zeroline=True,
    )
    preds_fig.update_layout(
        title=dict(text="GAM-Estimated LWS Dynamics for Time on Task", font=cnfg.TITLE_FONT),
        legend=dict(
            orientation="h",
            yanchor="middle", y=1,
            xanchor="right", x=0.95,
        ),
        margin=dict(t=40, b=40, l=40, r=10),
        template="plotly_white",
    )
    return preds_fig

In [5]:
plot_temporal_predictions(os.path.join(os.getcwd(), "R", "time-on-task_lws_predictions.csv"))